# LexCorpus — Exploratory Data Analysis & RAG Demo

This notebook covers:
1. Loading and inspecting raw ISAP data
2. Preprocessing overview (chunk size distribution)
3. A quick RAG demo using the Qdrant retriever
4. Evaluation of a few example questions

**Prerequisites**: Run the following before opening this notebook:
```bash
pip install -e .[dev]
python scripts/fetch_isap.py --year 2023 --output data/raw/
python scripts/preprocess.py --input data/raw/ --output data/processed/
python rag/ingest.py --input data/processed/chunks.jsonl
```

## 0. Setup

In [ ]:
import json
import os
import sys
from pathlib import Path
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np

# Add project root to path
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Project root:', PROJECT_ROOT)
print('Python:', sys.version)

## 1. Load Raw ISAP Data

In [ ]:
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
raw_files = sorted(RAW_DIR.glob('acts_*.jsonl'))
print(f'Found {len(raw_files)} raw JSONL files:')
for f in raw_files:
    line_count = sum(1 for _ in f.open())
    size_kb = f.stat().st_size // 1024
    print(f'  {f.name}: {line_count:,} acts, {size_kb:,} KB')

In [ ]:
# Load all raw records for analysis
raw_records = []
for f in raw_files:
    with f.open(encoding='utf-8') as fh:
        for line in fh:
            line = line.strip()
            if line:
                try:
                    raw_records.append(json.loads(line))
                except json.JSONDecodeError:
                    pass

print(f'Total raw records: {len(raw_records):,}')

# Show a sample record (without the full raw_text)
if raw_records:
    sample = {k: v for k, v in raw_records[0].items() if k != 'raw_text'}
    sample['raw_text_length'] = len(raw_records[0].get('raw_text', ''))
    print('\nSample record (fields):')
    for k, v in sample.items():
        print(f'  {k}: {v}')

In [ ]:
# Distribution of act types
type_counter = Counter(r.get('type', 'unknown') for r in raw_records)
print('Act types (top 10):')
for act_type, count in type_counter.most_common(10):
    print(f'  {act_type or "(empty)":30s}: {count:,}')

In [ ]:
# Distribution of act years
year_counter = Counter(r.get('year', 'unknown') for r in raw_records)
years_sorted = sorted(year_counter.items())

if years_sorted:
    years, counts = zip(*years_sorted)
    plt.figure(figsize=(12, 4))
    plt.bar([str(y) for y in years], counts, color='steelblue')
    plt.xlabel('Year')
    plt.ylabel('Number of acts')
    plt.title('ISAP Acts by Year')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

# Raw text length distribution
text_lengths = [len(r.get('raw_text', '')) for r in raw_records]
print(f'\nRaw text length stats (chars):')
print(f'  min   : {min(text_lengths) if text_lengths else 0:,}')
print(f'  median: {int(np.median(text_lengths)) if text_lengths else 0:,}')
print(f'  mean  : {int(np.mean(text_lengths)) if text_lengths else 0:,}')
print(f'  max   : {max(text_lengths) if text_lengths else 0:,}')
print(f'  empty : {sum(1 for l in text_lengths if l == 0):,}')

## 2. Preprocessed Chunks Analysis

In [ ]:
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
chunks_file = PROCESSED_DIR / 'chunks.jsonl'

chunks = []
if chunks_file.exists():
    with chunks_file.open(encoding='utf-8') as fh:
        for line in fh:
            line = line.strip()
            if line:
                try:
                    chunks.append(json.loads(line))
                except json.JSONDecodeError:
                    pass
    print(f'Loaded {len(chunks):,} chunks from {chunks_file}')
else:
    print(f'No chunks file found at {chunks_file}')
    print('Run: python scripts/preprocess.py --input data/raw/ --output data/processed/')

In [ ]:
if chunks:
    # Chunk token count distribution
    token_counts = [c.get('approx_tokens', len(c.get('text', '')) // 4) for c in chunks]

    plt.figure(figsize=(10, 4))
    plt.hist(token_counts, bins=50, color='darkorange', edgecolor='white')
    plt.axvline(512, color='red', linestyle='--', label='Target (512 tokens)')
    plt.xlabel('Approximate token count')
    plt.ylabel('Number of chunks')
    plt.title('Chunk Token Distribution')
    plt.legend()
    plt.tight_layout()
    plt.show()

    print(f'Chunk stats:')
    print(f'  Total chunks: {len(chunks):,}')
    print(f'  Mean tokens : {np.mean(token_counts):.0f}')
    print(f'  Median tokens: {np.median(token_counts):.0f}')
    print(f'  Chunks > 512 tokens: {sum(1 for t in token_counts if t > 512):,}')

    # Sample chunk
    print('\nSample chunk:')
    sample_chunk = chunks[0]
    print(f'  act_id: {sample_chunk["act_id"]}')
    print(f'  title: {sample_chunk["title"][:80]}')
    print(f'  year: {sample_chunk["year"]}')
    print(f'  chunk_index: {sample_chunk["chunk_index"]} / {sample_chunk["total_chunks"]}')
    print(f'  text preview: {sample_chunk["text"][:200]}...')

## 3. Quick RAG Demo

In [ ]:
# Initialize retriever (connects to Qdrant)
from rag.retriever import LegalRetriever

QDRANT_URL = os.getenv('QDRANT_URL', 'http://localhost:6333')

retriever = LegalRetriever(
    model_name='sdadas/mmlw-retrieval-roberta-large',
    collection='lexcorpus',
    qdrant_url=QDRANT_URL,
)
print('Retriever initialized. Embedding model:', retriever.model_name)

In [ ]:
# Example legal questions in Polish
example_questions = [
    'Jakie są prawa pracownika przy wypowiedzeniu umowy o pracę?',
    'Jakie kary grożą za prowadzenie pojazdu pod wpływem alkoholu?',
    'Co to jest skarga kasacyjna i kiedy można ją wnieść?',
    'Jakie są obowiązki pracodawcy w zakresie bhp?',
]

QUERY = example_questions[0]
print(f'Query: {QUERY}')

In [ ]:
# Retrieve top-5 relevant chunks
results = retriever.retrieve(QUERY, top_k=5)

print(f'Retrieved {len(results)} chunks:\n')
for i, r in enumerate(results, 1):
    print(f'[{i}] Score: {r.score:.4f}')
    print(f'     {r.citation()}')
    print(f'     {r.text[:200]}...')
    print()

In [ ]:
# Format context for LLM prompt
context = retriever.format_context(results, max_chars=3000)
print('=== CONTEXT FOR LLM ===')
print(context[:1500])
print('...' if len(context) > 1500 else '')

## 4. Answer Generation (Claude API fallback)

In [ ]:
import anthropic
from dotenv import load_dotenv

load_dotenv(PROJECT_ROOT / '.env')

ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY')

if not ANTHROPIC_API_KEY:
    print('Set ANTHROPIC_API_KEY in .env to run this cell')
else:
    client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

    SYSTEM_PROMPT = (
        'Jesteś ekspertem ds. polskiego prawa. Odpowiadasz na pytania prawne '
        'na podstawie podanych przepisów prawa polskiego. '
        'Udzielasz dokładnych, zwięzłych odpowiedzi w języku polskim. '
        'Zawsze powołujesz się na konkretne artykuły i akty prawne.'
    )

    user_message = (
        f'Na podstawie poniższych przepisów prawnych, odpowiedz na pytanie.\n\n'
        f'PRZEPISY:\n{context}\n\n'
        f'PYTANIE: {QUERY}\n\n'
        f'ODPOWIEDŹ:'
    )

    response = client.messages.create(
        model='claude-opus-4-5',
        max_tokens=1024,
        system=SYSTEM_PROMPT,
        messages=[{'role': 'user', 'content': user_message}],
    )

    answer = response.content[0].text
    print(f'=== QUESTION ===\n{QUERY}\n')
    print(f'=== ANSWER ===\n{answer}')

## 5. Similarity Score Visualization

In [ ]:
if results:
    # Plot score distribution for all retrieved results
    labels = [f'[{i+1}] {r.title[:30]}...' if len(r.title) > 30 else f'[{i+1}] {r.title}'
              for i, r in enumerate(results)]
    scores = [r.score for r in results]

    plt.figure(figsize=(10, 4))
    bars = plt.barh(labels, scores, color='steelblue')
    plt.xlabel('Cosine Similarity Score')
    plt.title(f'RAG Retrieval Scores\nQuery: "{QUERY[:60]}..."')
    plt.xlim(0, 1)
    for bar, score in zip(bars, scores):
        plt.text(score + 0.01, bar.get_y() + bar.get_height()/2,
                 f'{score:.4f}', va='center', fontsize=9)
    plt.tight_layout()
    plt.show()

## 6. Dataset Statistics

In [ ]:
from datasets import load_from_disk

DATASET_DIR = PROJECT_ROOT / 'data' / 'dataset'
instruct_path = DATASET_DIR / 'instruction'

if instruct_path.exists():
    ds = load_from_disk(str(instruct_path))
    print('Instruction-tuning dataset:')
    print(ds)
    print()
    print('Sample training record:')
    sample = ds['train'][0]
    for k, v in sample.items():
        val_str = str(v)[:120] + '...' if len(str(v)) > 120 else str(v)
        print(f'  {k}: {val_str}')
else:
    print(f'Dataset not found at {instruct_path}')
    print('Run: python scripts/build_dataset.py')